# Assignment 2.6 - Qwen-Image (Diffusers)

Infer Qwen-Image qua Diffusers:
- **2.1 Text2Img** voi `Qwen/Qwen-Image`
- **2.2 Image Editing / Img2Img** voi `Qwen/Qwen-Image-Edit`
- ComfyUI workflow cho Qwen-Image

> Qwen-Image ~20B -> **rat nang**: can GPU/dia lon (A100). Tren Colab free thuong KHONG du dia/VRAM
> -> dung ban quantized trong ComfyUI. Anh luu vao `output/`.

In [ ]:
import os, torch
from diffusers.utils import load_image

OUTPUT_DIR = "output"
os.makedirs(OUTPUT_DIR, exist_ok=True)
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.bfloat16 if device == "cuda" else torch.float32
print("device:", device, "| dtype:", dtype, "| torch:", torch.__version__)

# --- Patch 1: infer_schema (diffusers 0.38 dang ky custom-op annotation hoan; torch 2.4 khong tu suy ra) ---
import typing
import torch._library.infer_schema as _infer_schema_mod
_orig_infer_schema = _infer_schema_mod.infer_schema

def _patched_infer_schema(prototype_function, mutates_args):
    try:
        hints = typing.get_type_hints(
            prototype_function, globalns=getattr(prototype_function, "__globals__", {}))
    except Exception:
        hints = {}
    if hints:
        orig = dict(getattr(prototype_function, "__annotations__", {}))
        prototype_function.__annotations__ = {**orig, **hints}
        try:
            return _orig_infer_schema(prototype_function, mutates_args)
        finally:
            prototype_function.__annotations__ = orig
    return _orig_infer_schema(prototype_function, mutates_args)

_infer_schema_mod.infer_schema = _patched_infer_schema
try:
    import torch._custom_op.impl as _custom_impl
    _custom_impl.infer_schema = _patched_infer_schema
except Exception:
    pass

# --- Patch 2: scaled_dot_product_attention ---
# diffusers 0.38 goi F.scaled_dot_product_attention(..., enable_gqa=...), chi co tu torch 2.5.
# torch 2.4.x khong nhan -> bo arg nay.
from packaging import version
if version.parse(torch.__version__.split("+")[0]) < version.parse("2.5"):
    _orig_sdpa = torch.nn.functional.scaled_dot_product_attention
    def _sdpa_compat(*args, **kwargs):
        kwargs.pop("enable_gqa", None)
        return _orig_sdpa(*args, **kwargs)
    torch.nn.functional.scaled_dot_product_attention = _sdpa_compat
    print("patched SDPA (drop enable_gqa for torch < 2.5)")

print("patched torch for diffusers import")

In [ ]:
# (1) Dang nhap HF MA KHONG ghi token vao notebook.
#     Thu tu lay token: Colab Secrets -> file .env -> bien moi truong -> nhap tay.
import pathlib
from huggingface_hub import login

def _load_dotenv():
    """Doc file .env (tim o cwd va cac thu muc cha) roi nap vao os.environ."""
    here = pathlib.Path.cwd()
    for d in [here, *here.parents]:
        f = d / ".env"
        if f.is_file():
            for line in f.read_text(encoding="utf-8").splitlines():
                line = line.strip()
                if line and not line.startswith("#") and "=" in line:
                    k, v = line.split("=", 1)
                    os.environ.setdefault(k.strip(), v.strip().strip('"').strip("'"))
            return str(f)
    return None

def get_hf_token():
    # a) Colab Secrets (icon chia khoa ben trai Colab -> them HF_TOKEN, bat Notebook access)
    try:
        from google.colab import userdata
        tok = userdata.get("HF_TOKEN")
        if tok:
            return tok
    except Exception:
        pass
    # b) file .env (sao chep .env.example -> .env va dien token that)
    _load_dotenv()
    tok = os.environ.get("HF_TOKEN", "")
    if tok and not tok.startswith("hf_your"):     # bo qua gia tri placeholder
        return tok
    # c) nhap tay - an tren man hinh, khong luu vao file
    from getpass import getpass
    return getpass("Nhap HF token (https://huggingface.co/settings/tokens): ")

login(token=get_hf_token())
print("HF login OK")

# (2) Dung luong dia: Qwen-Image ~40GB. Kiem tra cho trong (dung cwd cho portable, tranh loi /root):
import shutil
free_gb = shutil.disk_usage(os.getcwd()).free / 1e9
print(f"Free disk: {free_gb:.1f} GB")
if free_gb < 50:
    print("!! Thieu dia cho Qwen-Image (~40GB). Don cache cu hoac dung A100/dia lon hon.")

## 2.1 Text2Img - `Qwen-Image`

In [ ]:
from diffusers import DiffusionPipeline

pipe = DiffusionPipeline.from_pretrained("Qwen/Qwen-Image", torch_dtype=dtype)
pipe.enable_model_cpu_offload()

prompt = "a coffee shop storefront with a neon sign that reads \"Flow Matching\", photorealistic"
image = pipe(
    prompt,
    width=1024, height=1024,
    num_inference_steps=50,
    true_cfg_scale=4.0,
    generator=torch.Generator("cpu").manual_seed(0),
).images[0]

image.save(os.path.join(OUTPUT_DIR, "qwen_t2i.png"))
print("saved output/qwen_t2i.png")
image

## 2.2 Image Editing / Img2Img - `Qwen-Image-Edit`

Manh ve chinh sua theo lenh: them/bot vat the, doi chu, doi phong cach.

In [ ]:
from diffusers import QwenImageEditPipeline
import gc

# Giai phong pipe Text2Img (~20B) truoc khi nap Edit, neu khong se OOM
try:
    del pipe
except NameError:
    pass
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

pipe_edit = QwenImageEditPipeline.from_pretrained("Qwen/Qwen-Image-Edit", torch_dtype=dtype)
pipe_edit.enable_model_cpu_offload()

edit_in = load_image(
    "https://raw.githubusercontent.com/CompVis/stable-diffusion/main/assets/stable-samples/img2img/sketch-mountains-input.jpg"
).resize((1024, 1024))

image = pipe_edit(
    image=edit_in,
    prompt="add a full moon in the sky and make it nighttime",
    num_inference_steps=40,
    true_cfg_scale=4.0,
    generator=torch.Generator("cpu").manual_seed(0),
).images[0]

image.save(os.path.join(OUTPUT_DIR, "qwen_edit.png"))
print("saved output/qwen_edit.png")
image

## ComfyUI - Workflow cho Qwen-Image

ComfyUI co san template: **Workflow -> Browse Templates -> Qwen-Image**.
- Node **Qwen-Image** cho Text2Img.
- Node **Qwen-Image-Edit** cho img2img/editing (nhan anh dau vao + prompt chinh sua).

**Tang chat luong:** Upscale (`UpscaleModelLoader` + `ImageUpscaleWithModel`), true-CFG hop ly.
**Tang toc:** dung ban **GGUF/fp8 quantized** (`UnetLoaderGGUF`) de chay GPU nho; giam steps; bat `--fast`.